# Spare-Part Demand Forecasting - NewX Services (Ref PM-PR-0027)
**Goal:** forecast spare-part demand per part x service centre x week, 4 weeks ahead, and beat a naive baseline by >= 20% on overstock and stockout events.

**Flow:** load -> clean -> EDA -> weekly panel -> features -> model vs baselines -> importance -> forecast -> recommendations.
Logic lives in `src/pipeline.py` (also used by `run.py` and the Streamlit app). Run this notebook from the repo root or `notebooks/`.

**Standalone:** the next two cells install packages and write `src/pipeline.py` (the full pipeline) so this notebook runs alone in Colab or Jupyter.
**Data access:** if `data/raw_service_data.csv` exists it is used. Otherwise the notebook asks for the MySQL host/user/password. No credentials are stored in this notebook.

In [ ]:
!pip install -q lightgbm sqlalchemy pymysql
import os; os.makedirs("src", exist_ok=True)

In [ ]:
%%writefile src/pipeline.py
"""Spare-part demand forecasting: load -> clean -> weekly panel -> features -> LightGBM -> 4-week forecast.

Direct multi-step design: every feature uses data at least H weeks old, so one model
forecasts all H weeks ahead without recursive prediction and without look-ahead leakage.
"""
import json
import os
import re
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd

H = 4  # forecast horizon (weeks) and minimum lag
TEST_WEEKS = 12  # last N observed weeks held out for evaluation
RAW_COLS = {
    "Invoice Date": "invoice_date",
    "Job Card Date": "job_card_date",
    "Business Partner Name": "partner",
    "Vehicle No.": "vehicle_no",
    "Vehicle Model": "vehicle_model",
    "Current KM Reading": "km",
    "INVOICE LINE TEXT": "line_text",
}
FEATURES = [
    "lag_4", "lag_5", "lag_6", "lag_8", "lag_12",
    "roll_mean_4", "roll_mean_8", "roll_mean_13", "roll_std_8",
    "km_lag", "km_bin", "jobs_lag",
    "week_of_year", "month", "woy_sin", "woy_cos",
    "item_c", "centre_c",
]


# ---------- load / clean ----------
def load_raw(csv=None, cache="data/raw_service_data.csv"):
    """MySQL if DB_HOST is set (credentials from env, never in code), else a CSV."""
    if os.getenv("DB_HOST"):
        from sqlalchemy import create_engine
        from sqlalchemy.engine import URL

        url = URL.create(
            "mysql+pymysql",
            username=os.environ["DB_USER"].strip(),
            password=os.environ["DB_PASSWORD"].strip(),
            host=os.environ["DB_HOST"].strip(),
            port=int(os.getenv("DB_PORT", 3306)),
            database=os.environ["DB_NAME"],
        )
        df = pd.read_sql(f"SELECT * FROM {os.getenv('DB_TABLE', 'service_data')}", create_engine(url))
        Path(cache).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(cache, index=False)
        return df
    path = csv or cache
    if not Path(path).exists():
        raise FileNotFoundError(f"{path} not found. Set DB_* env vars (see .env.example) or pass a CSV.")
    return pd.read_csv(path)


def norm_item(text):
    """'OIL FILTER - 4521' -> 'oil filter'. Digits/punctuation dropped, first 4 words kept."""
    return " ".join(re.sub(r"[^a-z ]", " ", str(text).lower()).split()[:4])


def clean(df):
    df = df.rename(columns=lambda c: c.strip()).rename(columns=RAW_COLS)
    for c in ("invoice_date", "job_card_date"):
        df[c] = pd.to_datetime(df[c], errors="coerce")
    df["invoice_date"] = df["invoice_date"].fillna(df["job_card_date"])
    df["km"] = pd.to_numeric(df["km"], errors="coerce")
    df = df.dropna(subset=["invoice_date", "line_text"]).copy()
    df["item"] = df["line_text"].map(norm_item)
    # repeated identical lines are kept: there is no quantity column, a repeat can mean qty > 1
    return df[df["item"] != ""].reset_index(drop=True)


# ---------- weekly panel ----------
def weekly_panel(df, centre_col=None, top_items=25, top_centres=5):
    """Full item x centre x week grid. demand = invoice lines per week (no qty column exists).

    centre_col: column used as 'service centre' ('partner' is the only proxy in the data).
    Returns (panel, last_observed_week). The panel includes H future weeks with demand = NaN.
    """
    df = df.copy()
    df["week"] = df["invoice_date"].dt.normalize() - pd.to_timedelta(df["invoice_date"].dt.weekday, unit="D")
    d0, d1 = df["invoice_date"].min().normalize(), df["invoice_date"].max().normalize()
    first = d0 - pd.Timedelta(days=d0.weekday())
    if d0 != first:
        first += pd.Timedelta(weeks=1)  # drop partial first week
    last = d1 - pd.Timedelta(days=d1.weekday())
    if d1 != last + pd.Timedelta(days=6):
        last -= pd.Timedelta(weeks=1)  # drop partial last week
    df = df[(df["week"] >= first) & (df["week"] <= last)]
    jobs = df.groupby("week").size()  # shop-wide volume, all items

    df["centre"] = df[centre_col].astype(str).str.strip() if centre_col else "ALL"
    df = df[df["item"].isin(df["item"].value_counts().head(top_items).index)]
    df = df[df["centre"].isin(df["centre"].value_counts().head(top_centres).index)]

    g = df.groupby(["item", "centre", "week"]).agg(demand=("item", "size"), km_mean=("km", "mean")).reset_index()
    weeks = pd.DataFrame({"week": pd.date_range(first, last + pd.Timedelta(weeks=H), freq="7D")})
    p = g[["item", "centre"]].drop_duplicates().merge(weeks, how="cross").merge(g, on=["item", "centre", "week"], how="left")
    p["demand"] = np.where(p["week"] <= last, p["demand"].fillna(0), np.nan)
    p["jobs"] = p["week"].map(jobs)
    return p.sort_values(["item", "centre", "week"]).reset_index(drop=True), last


# ---------- features ----------
def add_features(p):
    p = p.copy()
    g = p.groupby(["item", "centre"])
    for k in (H, H + 1, H + 2, H + 4, H + 8):
        p[f"lag_{k}"] = g["demand"].shift(k)
    for w in (4, 8, 13):
        p[f"roll_mean_{w}"] = g["demand"].transform(lambda s, w=w: s.shift(H).rolling(w, min_periods=1).mean())
    p["roll_std_8"] = g["demand"].transform(lambda s: s.shift(H).rolling(8, min_periods=2).std())
    p["km_lag"] = g["km_mean"].transform(lambda s: s.shift(H).ffill())
    p["km_bin"] = (p["km_lag"] // 10000).clip(upper=15)  # 10k-km service-interval bins
    wk_jobs = p.groupby("week")["jobs"].first().sort_index().shift(H)
    p["jobs_lag"] = p["week"].map(wk_jobs)
    p["week_of_year"] = p["week"].dt.isocalendar().week.astype(int)
    p["month"] = p["week"].dt.month
    p["woy_sin"] = np.sin(2 * np.pi * p["week_of_year"] / 52)
    p["woy_cos"] = np.cos(2 * np.pi * p["week_of_year"] / 52)
    p["item_c"] = p["item"].astype("category")
    p["centre_c"] = p["centre"].astype("category")
    return p


def split(p, last):
    """Time-based split. Returns (train, test, future)."""
    obs = p[p["demand"].notna()].dropna(subset=["lag_4"])
    cut = last - pd.Timedelta(weeks=TEST_WEEKS - 1)
    return obs[obs["week"] < cut], obs[obs["week"] >= cut], p[p["week"] > last]


# ---------- model / evaluation ----------
def fit(train):
    m = lgb.LGBMRegressor(
        objective="poisson", n_estimators=400, learning_rate=0.03, num_leaves=15,
        min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        random_state=0, verbose=-1,
    )
    return m.fit(train[FEATURES], train["demand"])


def evaluate(y, pred):
    """Order quantity = rounded forecast. overstock: ordered > needed; stockout: ordered < needed."""
    y, pred = np.asarray(y, float), np.asarray(pred, float)
    order = np.rint(pred)
    over, under = np.maximum(order - y, 0), np.maximum(y - order, 0)
    nz = y > 0
    return {
        "MAE": float(np.abs(y - pred).mean()),
        "RMSE": float(np.sqrt(((y - pred) ** 2).mean())),
        "WAPE": float(np.abs(y - pred).sum() / max(y.sum(), 1e-9)),
        "MAPE_nonzero": float((np.abs(y - pred)[nz] / y[nz]).mean()) if nz.any() else float("nan"),
        "overstock_events": int((over > 0).sum()),
        "stockout_events": int((under > 0).sum()),
        "overstock_units": float(over.sum()),
        "stockout_units": float(under.sum()),
    }


def compare(test, pred):
    rows = {
        "naive_lag4": evaluate(test["demand"], test["lag_4"]),  # same info set as the model
        "moving_avg_4": evaluate(test["demand"], test["roll_mean_4"]),
        "lightgbm": evaluate(test["demand"], pred),
    }
    tab = pd.DataFrame(rows).T
    n, m = tab.loc["naive_lag4"], tab.loc["lightgbm"]
    gain = {k: float(1 - m[k] / n[k]) if n[k] else float("nan") for k in
            ("overstock_events", "stockout_events", "overstock_units", "stockout_units")}
    return tab, gain


# ---------- end to end ----------
def run(df, centre_col=None, out="outputs", models="models", top_items=25, top_centres=5):
    Path(out).mkdir(exist_ok=True, parents=True)
    Path(models).mkdir(exist_ok=True, parents=True)
    panel, last = weekly_panel(df, centre_col, top_items, top_centres)
    p = add_features(panel)
    train, test, future = split(p, last)

    tab, gain = compare(test, fit(train).predict(test[FEATURES]))
    final = fit(pd.concat([train, test]))  # refit on all observed data for the real forecast
    joblib.dump({"model": final, "features": FEATURES, "horizon": H}, f"{models}/demand_model.joblib")

    fc = future[["item", "centre", "week"]].copy()
    fc["predicted_demand"] = np.clip(final.predict(future[FEATURES]), 0, None).round(2)
    fc.rename(columns={"item": "part_name", "centre": "service_centre"}).to_csv(f"{out}/forecast.csv", index=False)

    tp = test[["item", "centre", "week", "demand", "lag_4"]].copy()
    tp["pred"] = fit(train).predict(test[FEATURES])
    tp.to_csv(f"{out}/test_predictions.csv", index=False)
    p[p["demand"].notna()][["item", "centre", "week", "demand"]].to_csv(f"{out}/weekly_history.csv", index=False)
    pd.DataFrame({"feature": FEATURES, "gain": final.booster_.feature_importance("gain")}) \
        .sort_values("gain", ascending=False).to_csv(f"{out}/feature_importance.csv", index=False)

    metrics = {
        "horizon_weeks": H, "test_weeks": TEST_WEEKS, "last_observed_week": str(last.date()),
        "n_items": int(p["item"].nunique()), "n_centres": int(p["centre"].nunique()),
        "n_train_rows": len(train), "n_test_rows": len(test),
        "table": tab.to_dict("index"), "improvement_vs_naive": gain,
        "target_met": bool(all(v >= 0.20 for k, v in gain.items() if k.endswith("events"))),
    }
    Path(f"{out}/metrics.json").write_text(json.dumps(metrics, indent=2))
    return metrics


In [ ]:
import os, getpass
os.makedirs("src", exist_ok=True); open("src/__init__.py", "a").close()
if not os.path.exists("data/raw_service_data.csv") and not os.getenv("DB_HOST"):
    os.environ.update(DB_HOST=input("DB host: "), DB_PORT="3306", DB_NAME="project_service_data", DB_TABLE="service_data",
                      DB_USER=input("DB user: "), DB_PASSWORD=getpass.getpass("DB password: "))

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src import pipeline as P
sns.set_theme(style="whitegrid")

CENTRE_COL = None      # the table has no service-centre column. Set "partner" to use Business Partner Name as a proxy.
TOP_ITEMS, TOP_CENTRES = 25, 5

## 1. Load and clean

In [ ]:
raw = P.load_raw()
print(raw.shape); raw.head()

In [ ]:
df = P.clean(raw)
print(f"{len(raw)} raw rows -> {len(df)} after cleaning")
print(df.isna().mean().round(3).to_string())
df[["invoice_date","vehicle_model","km","line_text","item"]].head(10)

## 2. EDA

In [ ]:
print("date range:", df.invoice_date.min().date(), "->", df.invoice_date.max().date())
print("vehicles:", df.vehicle_no.nunique(), "| models:", df.vehicle_model.nunique(), "| partners:", df.partner.nunique(), "| distinct items:", df.item.nunique())
df.km.describe().round(0)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
df.set_index("invoice_date").resample("MS").size().plot(ax=ax[0], title="Invoice lines per month")
df.item.value_counts().head(20).sort_values().plot.barh(ax=ax[1], title="Top 20 items by line count")
plt.tight_layout()

**Seasonality** - share of each top item's lines by calendar month (a flat row = no seasonality).

In [ ]:
top = df.item.value_counts().head(12).index
s = df[df.item.isin(top)].assign(month=lambda d: d.invoice_date.dt.month)
sm = pd.crosstab(s.item, s.month, normalize="index")
plt.figure(figsize=(10, 5)); sns.heatmap(sm, cmap="YlOrRd", annot=True, fmt=".2f"); plt.title("Monthly share of demand by item")

**Demand by vehicle model and odometer band** (10,000-km service-interval bins).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
df.vehicle_model.value_counts().head(10).sort_values().plot.barh(ax=ax[0], title="Lines by vehicle model")
df.assign(km_bin=(df.km // 10000).clip(upper=15) * 10).groupby("km_bin").size().plot.bar(ax=ax[1], title="Lines by odometer band (k km)")
plt.tight_layout()

In [ ]:
topm = df.vehicle_model.value_counts().head(6).index
ct = pd.crosstab(df[df.vehicle_model.isin(topm)].vehicle_model, df[df.vehicle_model.isin(topm)].item.where(lambda s: s.isin(top), "other"), normalize="index")
ct.plot.bar(stacked=True, figsize=(11, 4), colormap="tab20", title="Item mix by vehicle model").legend(bbox_to_anchor=(1, 1))

## 3. Weekly panel (item x centre x week)
Demand = invoice lines per week: there is no quantity column. Partial first/last weeks are dropped; missing weeks are zeros.

In [ ]:
panel, last = P.weekly_panel(df, CENTRE_COL, TOP_ITEMS, TOP_CENTRES)
print("last observed full week:", last.date(), "| series:", panel[["item","centre"]].drop_duplicates().shape[0])
obs = panel[panel.demand.notna()]
print("zero-demand share (intermittency):", round((obs.demand == 0).mean(), 3))
obs.groupby("week").demand.sum().plot(figsize=(11, 3), title="Total weekly demand, modelled items")

## 4. Features
Every feature uses data at least **H = 4 weeks old** (lags 4/5/6/8/12, rolling mean/std shifted by 4, last-known odometer + 10k-km bin, shop-wide job volume, week-of-year sin/cos). One direct model therefore forecasts weeks T+1..T+4 with no recursion and no leakage (verified in `tests/`).

In [ ]:
feat = P.add_features(panel)
train, test, future = P.split(feat, last)
print(len(train), "train rows |", len(test), "test rows (last", P.TEST_WEEKS, "weeks) |", len(future), "future rows")
feat[P.FEATURES].describe().T.round(2)

## 5. Model vs baselines
Baselines: **naive** = demand 4 weeks ago (same information as the model), **moving average** of 4 weeks. Model: LightGBM, Poisson loss (counts, many zeros). Order quantity = rounded forecast; *overstock* = ordered above need, *stockout* = ordered below need. MAPE is undefined on zero weeks so it is computed on non-zero weeks only.

In [ ]:
model = P.fit(train)
pred = model.predict(test[P.FEATURES])
tab, gain = P.compare(test, pred)
display(tab.round(3))
print({k: f"{v:.1%}" for k, v in gain.items()})
print("Target (>=20% fewer overstock AND stockout events vs naive):", all(gain[k] >= .2 for k in ("overstock_events","stockout_events")))

In [ ]:
t = test.assign(pred=pred).groupby("week")[["demand", "lag_4", "pred"]].sum()
t.rename(columns={"demand":"actual","lag_4":"naive","pred":"LightGBM"}).plot(figsize=(11, 3.5), title="Hold-out: total weekly demand")

In [ ]:
sel = test.assign(pred=pred).groupby("item").apply(lambda d: pd.Series({
    "actual": d.demand.sum(), "WAPE_model": (d.demand - d.pred).abs().sum() / max(d.demand.sum(), 1),
    "WAPE_naive": (d.demand - d.lag_4).abs().sum() / max(d.demand.sum(), 1)})).sort_values("actual", ascending=False)
sel.round(3).head(15)

## 6. Feature importance

In [ ]:
imp = pd.Series(model.booster_.feature_importance("gain"), index=P.FEATURES).sort_values()
imp.plot.barh(figsize=(8, 5), title="LightGBM gain importance")

## 7. Final model, forecast and artifacts
Refits on all observed weeks, saves `models/demand_model.joblib`, `outputs/forecast.csv` (part_name x service_centre x week -> predicted_demand) and the other outputs.

In [ ]:
m = P.run(df, CENTRE_COL, top_items=TOP_ITEMS, top_centres=TOP_CENTRES)
fc = pd.read_csv("outputs/forecast.csv", parse_dates=["week"])
display(fc.head(12))
fc.groupby("week").predicted_demand.sum().round(1)

In [ ]:
pn = fc.groupby("part_name").predicted_demand.sum().sort_values(ascending=False).head(10)
pn.sort_values().plot.barh(figsize=(8, 4), title="Top 10 parts: forecast demand, next 4 weeks")

## 8. Business recommendations
Fill in after reading the results above (numbers differ on the real data):
1. **Order from `forecast.csv`** weekly; start with the highest-volume parts (chart above).
2. **Safety stock:** add a buffer on parts where a stockout costs more than holding; the over/under trade-off table gives the dial.
3. **Seasonal parts:** pre-stock ahead of the months the seasonality heatmap flags.
4. **Parts with high WAPE** (per-item table): keep manual review; intermittent demand is inherently noisy.
5. **Data gaps to close:** quantity per line, part number, service-centre ID. They are the largest accuracy and relevance limits today.
6. **Operate:** retrain weekly with `python run.py`, track WAPE, view forecasts in `streamlit run app.py`.

**Limitations:** demand counts lines, not units; centre dimension is a proxy; labour and parts share one text field; ARIMA/Prophet/LSTM not run (one global gradient-boosted model chosen for simplicity).